# Marketing Funnel Master Table

This notebook builds a one-row-per-lead table from the marketing funnel CSV files.

Goals:
- inspect the two source files
- join leads to closed deals on `mql_id`
- derive conversion and time-to-win metrics

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)

PROJECT_DIR = Path(r"D:/Data_visualization_design/ecommerce-visual-analytics/data-visualization")
MARKETING_DIR = Path(r"D:/Data_visualization_design/ecommerce-visual-analytics/data/Marketing Funnel")
OUTPUT_PATH = PROJECT_DIR / "marketing_funnel_master_table.csv"

mql = pd.read_csv(MARKETING_DIR / "marketing_qualified_leads_dataset.csv", parse_dates=["first_contact_date"])
deals = pd.read_csv(MARKETING_DIR / "closed_deals_dataset.csv", parse_dates=["won_date"])

print(f"MQL rows: {len(mql):,}, columns: {len(mql.columns)}")
print(f"Closed deals rows: {len(deals):,}, columns: {len(deals.columns)}")
display(mql.head())
display(deals.head())

MQL rows: 8,000, columns: 4
Closed deals rows: 842, columns: 14


,mql_id,first_contact_date,landing_page_id,origin
0,dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social
1,8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search
2,b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search
3,6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email
4,5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search


,mql_id,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue
0,5420aad7fec3549a85876ba1c529bd84,2c43fb513632d29b3b58df74816f1b06,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2018-02-26 19:58:54,pet,online_medium,cat,NaN,NaN,NaN,reseller,NaN,0.0
1,a555fb36b9368110ede0f043dfc3b9a0,bbb7d7893a450660432ea6652310ebb7,09285259593c61296eef10c734121d5b,d3d1e91a157ea7f90548eef82f1955e3,2018-05-08 20:17:59,car_accessories,industry,eagle,NaN,NaN,NaN,reseller,NaN,0.0
2,327174d3648a2d047e8940d7d15204ca,612170e34b97004b3ba37eae81836b4c,b90f87164b5f8c2cfa5c8572834dbe3f,6565aa9ce3178a5caf6171827af3a9ba,2018-06-05 17:27:23,home_appliances,online_big,cat,NaN,NaN,NaN,reseller,NaN,0.0
3,f5fee8f7da74f4887f5bcae2bafb6dd6,21e1781e36faf92725dde4730a88ca0f,56bf83c4bb35763a51c2baab501b4c67,d3d1e91a157ea7f90548eef82f1955e3,2018-01-17 13:51:03,food_drink,online_small,NaN,NaN,NaN,NaN,reseller,NaN,0.0
4,ffe640179b554e295c167a2f6be528e0,ed8cb7b190ceb6067227478e48cf8dde,4b339f9567d060bcea4f5136b9f5949e,d3d1e91a157ea7f90548eef82f1955e3,2018-07-03 20:17:45,home_appliances,industry,wolf,NaN,NaN,NaN,manufacturer,NaN,0.0


In [2]:
marketing_funnel_master_table = mql.merge(deals, on="mql_id", how="left", suffixes=("", "_deal"))

marketing_funnel_master_table["is_converted_flag"] = marketing_funnel_master_table["seller_id"].notna()
marketing_funnel_master_table["time_to_win_days"] = (
    marketing_funnel_master_table["won_date"] - marketing_funnel_master_table["first_contact_date"]
).dt.days
marketing_funnel_master_table["lead_age_days"] = (
    pd.Timestamp.today().normalize() - marketing_funnel_master_table["first_contact_date"]
).dt.days
marketing_funnel_master_table["days_to_first_contact_missing_flag"] = marketing_funnel_master_table["first_contact_date"].isna()
marketing_funnel_master_table["won_date_missing_flag"] = marketing_funnel_master_table["won_date"].isna()

print(f"Marketing funnel master rows: {len(marketing_funnel_master_table):,}")
print(f"Duplicate mql_id rows: {marketing_funnel_master_table['mql_id'].duplicated().sum()}")
display(marketing_funnel_master_table.head())

Marketing funnel master rows: 8,000
Duplicate mql_id rows: 0


,mql_id,first_contact_date,landing_page_id,origin,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue,is_converted_flag,time_to_win_days,lead_age_days,days_to_first_contact_missing_flag,won_date_missing_flag
0,dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,3111,False,True
1,8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,3215,False,True
2,b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,3062,False,True
3,6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,3121,False,True
4,5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search,2c43fb513632d29b3b58df74816f1b06,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2018-02-26 19:58:54,pet,online_medium,cat,NaN,NaN,NaN,reseller,NaN,0.0,True,5.0,3091,False,False


In [3]:
expected_rows = len(mql)
actual_rows = len(marketing_funnel_master_table)

print(f"Expected MQL rows: {expected_rows:,}")
print(f"Actual master rows: {actual_rows:,}")
print(f"Duplicate mql_id rows: {marketing_funnel_master_table['mql_id'].duplicated().sum()}")
print("Missing values per column (top 15):")
display(marketing_funnel_master_table.isna().sum().sort_values(ascending=False).head(15))

marketing_funnel_master_table.to_csv(OUTPUT_PATH, index=False)
print(f"Marketing Funnel Master Table saved to: {OUTPUT_PATH}")
display(marketing_funnel_master_table.head())

Expected MQL rows: 8,000
Actual master rows: 8,000
Duplicate mql_id rows: 0
Missing values per column (top 15):


has_company                      7937
has_gtin                         7936
average_stock                    7934
declared_product_catalog_size    7931
lead_behaviour_profile           7335
business_type                    7168
lead_type                        7164
business_segment                 7159
time_to_win_days                 7158
won_date                         7158
sr_id                            7158
sdr_id                           7158
seller_id                        7158
declared_monthly_revenue         7158
origin                             60
dtype: int64

Marketing Funnel Master Table saved to: D:\Data_visualization_design\ecommerce-visual-analytics\data-visualization\marketing_funnel_master_table.csv


,mql_id,first_contact_date,landing_page_id,origin,seller_id,sdr_id,sr_id,won_date,business_segment,lead_type,lead_behaviour_profile,has_company,has_gtin,average_stock,business_type,declared_product_catalog_size,declared_monthly_revenue,is_converted_flag,time_to_win_days,lead_age_days,days_to_first_contact_missing_flag,won_date_missing_flag
0,dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,3111,False,True
1,8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,3215,False,True
2,b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,3062,False,True
3,6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,3121,False,True
4,5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search,2c43fb513632d29b3b58df74816f1b06,a8387c01a09e99ce014107505b92388c,4ef15afb4b2723d8f3d81e51ec7afefe,2018-02-26 19:58:54,pet,online_medium,cat,NaN,NaN,NaN,reseller,NaN,0.0,True,5.0,3091,False,False
